# 01 — Dữ liệu, EDA và tiền xử lý

Mục tiêu: dự đoán giá nhà trung vị của khu vực, phục vụ bài thực hành regression. Dùng California Housing theo thiết kế; ví dụ PDF dùng Boston Housing. Chia **80/10/10 trước tiền xử lý và tạo đặc trưng**, seed 42. EDA và các thống kê học chỉ dùng train; test giữ cho notebook 03.

Bảng và hình được lưu trong output của notebook này. `artifacts/` chỉ chứa dữ liệu thô đã chia và bộ biến đổi để bàn giao cho notebook tiếp theo; không xuất kết quả ra `results/` ở thư mục gốc.

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'src' / 'config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_DIR = ROOT / 'notebooks' / 'artifacts'
TARGET = 'median_house_value'
SEED = 42

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_files(manifest):
    for name, expected in manifest['files'].items():
        if file_hash(ARTIFACT_DIR / name) != expected:
            raise ValueError(f'Artifact mismatch: {name}. Rerun notebooks in order.')

from src.data.loader import load_housing_data, fetch_dataset_info
from src.data.preprocessing import split_data_8_1_1, fit_preprocessor, transform_features, save_processed_splits
from src.models.persistence import save_preprocessor, load_preprocessor
%matplotlib inline

## 1. Nạp và chia dữ liệu thô
Loader chỉ xác thực schema, numeric values và target; không điền khuyết, chuẩn hóa hay tạo ratios. Phân tầng theo phân vị target nhằm giữ phân bố giá tương đối cân bằng; nếu bins không khả thi, chương trình báo lỗi.

In [ ]:
data, target = load_housing_data(ROOT / 'data' / 'raw' / 'housing.csv')
splits = split_data_8_1_1(data.drop(columns=target), data[target], random_state=SEED)
X_train, X_val, X_test, y_train, y_val, y_test = splits
# Xác nhận độc lập trước khi CSV làm mất index gốc.
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0
assert len(X_train) + len(X_val) + len(X_test) == len(data)
assert all(X.index.equals(y.index) for X, y in zip(splits[:3], splits[3:]))
display(pd.DataFrame({'Split': ['Train', 'Validation', 'Test'],
                      'Rows': [len(X_train), len(X_val), len(X_test)],
                      'Fraction': [len(X_train)/len(data), len(X_val)/len(data), len(X_test)/len(data)]}))
print('Đã chia dữ liệu thô; chưa fit bất kỳ phép biến đổi nào.')


## 2. EDA chỉ trên train
Target giữ USD; không tự loại các dòng đạt trần giá hoặc nhóm hiếm. Thông tin target là median của khu vực, không phải giá giao dịch từng căn nhà.

In [ ]:
train_frame = X_train.assign(**{TARGET: y_train})
profile = fetch_dataset_info(train_frame, TARGET)
display(train_frame.head())
display(pd.DataFrame({'Missing': train_frame.isna().sum(), 'Dtype': train_frame.dtypes.astype(str)}))
display(train_frame.describe(include='all').T)
display(X_train['ocean_proximity'].value_counts(dropna=False).rename('Count').to_frame())
print(f"Số mẫu train đạt trần 500001 USD: {(y_train == 500001).sum()}")
print(f"Số dòng trùng hoàn toàn trong train: {profile['duplicate_rows']}")


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=40, color='steelblue', edgecolor='white')
axes[0].set(title='Train target distribution', xlabel='Median house value (USD)', ylabel='Count')
axes[1].scatter(X_train['median_income'], y_train, s=7, alpha=0.15)
axes[1].set(title='Train income vs. target', xlabel='Median income', ylabel='Median house value (USD)')
figure.tight_layout()
plt.show()
correlation = train_frame.select_dtypes(include='number').corr()
display(correlation[TARGET].sort_values(ascending=False).rename('Pearson_Correlation').to_frame())

## 3. Fit preprocessing sau split
Nhánh numeric: **median imputation → feature engineering tùy chọn → StandardScaler**. Nhánh categorical: **most-frequent imputation → OneHotEncoder** với unknown category thành vector 0. Tất cả tham số được học từ train.

Lần thực nghiệm chuẩn giữ `APPLY_LOG=False`, `CREATE_INTERACTIONS=False`. Module feature đã triển khai và kiểm thử; notebook 02 có đối chiếu riêng trên validation sau baseline. Nếu bật ratios, chúng được tính từ số gốc đã impute trước log, mẫu số phải dương. Không dùng target để tạo đặc trưng.

In [ ]:
APPLY_LOG = False
CREATE_INTERACTIONS = False
preprocessor = fit_preprocessor(X_train, apply_log=APPLY_LOG,
                                 create_interactions=CREATE_INTERACTIONS)
train_features = transform_features(preprocessor, X_train)
val_features = transform_features(preprocessor, X_val)
# Không cần biến đổi hoặc khám phá test ở giai đoạn này.
assert np.isfinite(train_features).all() and np.isfinite(val_features).all()
assert train_features.shape[1] == val_features.shape[1]
display(pd.DataFrame(train_features[:5], columns=preprocessor.get_feature_names_out()))
numeric_pipeline = preprocessor.named_transformers_['numeric']
display(pd.DataFrame({'Feature': numeric_pipeline.named_steps['imputer'].feature_names_in_,
                      'Train_Median': numeric_pipeline.named_steps['imputer'].statistics_}))
print(f'Số chiều đầu vào mô hình: {train_features.shape[1]}')


## 4. Bàn giao và kiểm tra tính nhất quán
CSV giữ nguyên missing values và USD target. Preprocessor chứa toàn bộ imputer, feature transformer, scaler và encoder. Hash phát hiện dữ liệu hoặc artifact bị thay đổi giữa các notebook; chạy lại 01 thì phải chạy lại 02 trước 03.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
save_processed_splits(*splits, target_col=TARGET, output_dir=ARTIFACT_DIR)
save_preprocessor(preprocessor, ARTIFACT_DIR / 'preprocessor.joblib')
manifest = {'seed': SEED, 'apply_log': APPLY_LOG, 'create_interactions': CREATE_INTERACTIONS,
            'raw_sha256': file_hash(ROOT / 'data' / 'raw' / 'housing.csv'),
            'files': {name: file_hash(ARTIFACT_DIR / name) for name in
                      ['train.csv', 'val.csv', 'test.csv', 'preprocessor.joblib']}}
(ARTIFACT_DIR / 'preparation_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
restored = load_preprocessor(ARTIFACT_DIR / 'preprocessor.joblib')
np.testing.assert_allclose(transform_features(restored, X_val), val_features)
print('Đã lưu dữ liệu thô đã chia và bộ tiền xử lý trong notebooks/artifacts; kiểm tra nạp lại đạt.')


## Kết luận giai đoạn
Đã kiểm tra các tập không giao nhau, X/y đồng bộ và preprocessing chỉ fit trên train. Quan hệ income–target quan sát được không chứng minh nhân quả. Nhóm hiếm và target capping được giữ nguyên để phân tích hạn chế khi đánh giá cuối.